# Figure S7 - true-genealogy homoplasy and variant confusability

Across-run version built from the true simulation genealogies (`run-out.branches`). Panel **A** is the origin-count CDF restricted to *established* origins (those whose clade retains at least `MIN_PROGENY` sampled infections) -- the lineages variant assignment operates on; among them independent recurrence is rare. Panel **B** is the shared-substitution minus null co-assignment per method (unfiltered). Each simulation is a faint curve; the representative run is bold.


## Imports and parameters

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

HERE = Path.cwd()
sys.path.append(str(HERE.parent / 'scripts'))

from plot_truetree_homoplasy import ESTABLISHED_PROGENY, build_pooled_figure
from plot_mutation_homoplasy import REPRESENTATIVE_RUN
from antigentools.supplement_style import apply_supplement_style

apply_supplement_style()

BATCH = '2026-07-04-reviewer-runs'
AGG_DIR = HERE.parent / 'results' / 'aggregated' / BATCH
CAND_PATH = HERE.parent / 'data' / BATCH / 'antigen-outputs' / 'candidate_runs.csv'
FIG_DIR = HERE.parent.parent / 'antigen-tex' / 'reviews' / 'round1' / 'figures'
FIG_PREFIX = 'figureS7_truetree_homoplasy'

# 'Established' progeny threshold for panel A. The sweep emits every threshold in
# plot_truetree_homoplasy.PROGENY_THRESHOLDS, so this can be retuned here without
# re-running the sweep. Defaults to the module's ESTABLISHED_PROGENY.
MIN_PROGENY = ESTABLISHED_PROGENY
print(f'panel A restricted to origins with >= {MIN_PROGENY} sampled infections')

## Load the sweep output and restrict to the flu-like candidate runs

In [ ]:
recurrence = pd.read_csv(AGG_DIR / 'truetree_recurrence_by_run.csv')
origin_counts = pd.read_csv(AGG_DIR / 'truetree_origin_counts_by_run.csv')
confusability = pd.read_csv(AGG_DIR / 'truetree_confusability_by_run.csv')

cand = pd.read_csv(CAND_PATH)


def config_from_path(p):
    m = re.search(r'simulations/([^/]+)/run_', p)
    assert m is not None, f'cannot parse config from candidate path: {p!r}'
    return m.group(1)


cand['config'] = cand['path'].map(config_from_path)
cand_keys = set(zip(cand['config'], cand['run'].astype(int)))


def keep_candidates(df, label):
    keys = list(zip(df['config'], df['run'].astype(int)))
    kept = df[[k in cand_keys for k in keys]].copy()
    assert not kept.empty, f'no candidate runs survived the join for {label}'
    print(f'{label}: {len(kept)} rows from {kept.groupby(["config", "run"]).ngroups} candidate runs')
    return kept


recurrence = keep_candidates(recurrence, 'per-run recurrence')
origin_counts = keep_candidates(origin_counts, 'origin-count ECDF')
confusability = keep_candidates(confusability, 'confusability')

assert MIN_PROGENY in set(origin_counts['min_progeny']), \
    f'MIN_PROGENY={MIN_PROGENY} not among emitted thresholds {sorted(set(origin_counts["min_progeny"]))}'

## Figure S7

In [ ]:
fig = build_pooled_figure(origin_counts, confusability, REPRESENTATIVE_RUN, min_progeny=MIN_PROGENY)
plt.show()

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
for suffix in ('pdf', 'png'):
    path = FIG_DIR / f'{FIG_PREFIX}.{suffix}'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Wrote {path}')

**Figure S7.** True-genealogy homoplasy and variant confusability across all flu-like candidate `antigen-prime` simulations. Each faint curve is one simulation; the representative run is bold.

**(A)** Cumulative fraction of substitutions of each site class arising on at most X separate branches, restricted to origins whose clade retains at least `MIN_PROGENY` sampled infections. Among the established lineages that variant assignment operates on, independent recurrence of the same substitution is rare.

**(B)** For every pair of independent origins of one substitution, the change in the probability that the two origins' descendants receive the same variant label, relative to a matched null of origin pairs from different substitutions at the same background distance. Zero means sharing a substitution adds no co-assignment beyond background similarity.

## Numbers quoted in the response letter

Pooled across candidate runs, weighting by substitution counts. Reported both unfiltered (min_progeny 1) and among established lineages (min_progeny = MIN_PROGENY).

In [ ]:
def pooled_unfiltered(site):
    return recurrence[f'{site}_n_recurrent'].sum() / recurrence[f'{site}_n'].sum()


def pooled_established(site):
    oc = origin_counts[(origin_counts['min_progeny'] == MIN_PROGENY)
                       & (origin_counts['site_class'] == site)]
    # one row per (run, x); the per-run n and its recurring fraction come from the
    # x==1 row (n_substitutions) and the tail beyond x==1.
    per_run = oc.groupby(['config', 'run'])
    n_tot = rec_tot = 0
    for _key, g in per_run:
        n = int(g['n_substitutions'].iloc[0])
        frac_le1 = float(g[g['x'] == 1]['cumulative_fraction'].iloc[0])
        n_tot += n
        rec_tot += round(n * (1 - frac_le1))
    return rec_tot / n_tot if n_tot else float('nan'), n_tot


for site in ('epitope', 'non_epitope'):
    est_rate, est_n = pooled_established(site)
    print(f'{site:12s} unfiltered {pooled_unfiltered(site):6.1%}   '
          f'established(>={MIN_PROGENY} inf) {est_rate:6.1%} of {est_n} substitutions')